# TextSense AI - Intelligent Text Analytics & Predictor

## 1. Import Libraries

In [ ]:
import pandas as pd

In [ ]:
import os

print(os.listdir("/kaggle/input"))

In [ ]:
print(os.listdir("/kaggle/input/datasets"))

In [ ]:
print(os.listdir("/kaggle/input/datasets/uciml"))

In [ ]:
print(os.listdir("/kaggle/input/datasets/uciml/sms-spam-collection-dataset"))

## 2. Load Dataset

In [ ]:
df = pd.read_csv(
    "/kaggle/input/datasets/uciml/sms-spam-collection-dataset/spam.csv",
    encoding="latin-1"
)

In [ ]:
print(df.head())

In [ ]:
print(df.shape)

In [ ]:
print(df.columns)

In [ ]:
df = df[["v1", "v2"]]

df.columns = ["label", "message"]

print(df.head())

## 3. Data Cleaning

The original dataset contains unused columns and is reduced to the two required columns:

- **label** — HAM or SPAM
- **message** — SMS text

Basic dataset information, class distribution and missing values are checked before further processing.

In [ ]:
print(df.info())

In [ ]:
print(df["label"].value_counts())

In [ ]:
print(df.isnull().sum())

In [ ]:
df = df.copy()

df["message_length"] = df["message"].apply(len)

In [ ]:
print(df[["message", "message_length"]].head())

In [ ]:
print(df.groupby("label")["message_length"].mean())

In [ ]:
print(df.duplicated().sum())

## 4. Exploratory Data Analysis

Exploratory Data Analysis was performed to understand the dataset structure, class distribution and message-length patterns.

The dataset contains substantially more HAM messages than SPAM messages.

Message length was also examined to understand whether spam messages tend to differ in length from normal messages.

In [ ]:
print("Class Distribution:")
print(df["label"].value_counts())

print("\nAverage Message Length:")
print(df.groupby("label")["message_length"].mean())

print("\nDuplicate Records:")
print(df.duplicated().sum())

## 5. Duplicate Removal

In [ ]:
print(df[df.duplicated()].head())

In [ ]:
df = df.drop_duplicates()

In [ ]:
print(df.shape)

In [ ]:
print(df.duplicated().sum())

### After Duplicate Removal

After removing duplicate records:

- **Total messages:** 5,169
- **HAM messages:** 4,516
- **SPAM messages:** 653
- **Duplicates remaining:** 0

## 6. Label Encoding

Machine learning models require numerical target labels.

The SMS labels are encoded as:

- **HAM = 0**
- **SPAM = 1**

In [ ]:
df["label_num"] = df["label"].map({
    "ham": 0,
    "spam": 1
})

In [ ]:
print(df[["label", "label_num"]].head())

In [ ]:
import re

def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)
    return text

In [ ]:
print(clean_text("WIN a FREE prize!!!"))

In [ ]:
df["clean_message"] = df["message"].apply(clean_text)

In [ ]:
df["clean_message"] = ...

In [ ]:
print(df[["message", "clean_message"]].head())

In [ ]:
print(df["label"].value_counts())

In [ ]:
print(df["label_num"].value_counts())

## 7. Text Preprocessing

The SMS messages are cleaned before converting them into numerical features.

The preprocessing converts text to lowercase and removes non-alphabetic characters.

In [ ]:
X = df["clean_message"]
y = df["label_num"]

print(X.head())
print(y.head())

## 8. Train/Test Split

The cleaned dataset is divided into training and testing sets using an 80/20 split.

- **Training samples:** 4,135
- **Testing samples:** 1,034
- **Test size:** 20%
- **Random state:** 42
- **Stratification:** Used

Stratification helps maintain a similar HAM/SPAM class distribution in both sets.

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

In [ ]:
print("Training labels:")
print(y_train.value_counts())

print("Testing labels:")
print(y_test.value_counts())

## 9. TF-IDF Vectorization

TF-IDF (Term Frequency-Inverse Document Frequency) is used to convert SMS text into numerical features.

The baseline representation uses **unigrams**, meaning individual words are used as features.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [ ]:
vectorizer = TfidfVectorizer()

In [ ]:
print(df["clean_message"].apply(type).value_counts())

In [ ]:
df["clean_message"] = df["message"].apply(clean_text)

In [ ]:
print(df["clean_message"].apply(type).value_counts())

In [ ]:
X = df["clean_message"]
y = df["label_num"]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
X_train_tfidf = vectorizer.fit_transform(X_train)

In [ ]:
X_test_tfidf = vectorizer.transform(X_test)

In [ ]:
print(X_train_tfidf.shape)

In [ ]:
print(X_test_tfidf.shape)

In [ ]:
print(vectorizer.get_feature_names_out()[:20])

In [ ]:
print(type(X_train_tfidf))

## 10. Naive Bayes Model

A Multinomial Naive Bayes classifier is trained using the TF-IDF features.

This model is used as a baseline machine learning approach for SMS spam classification.

In [ ]:
from sklearn.naive_bayes import MultinomialNB

In [ ]:
nb_model = MultinomialNB()

In [ ]:
nb_model.fit(X_train_tfidf, y_train)

In [ ]:
nb_predictions = nb_model.predict(X_test_tfidf)

In [ ]:
print(nb_predictions[:20])

In [ ]:
from sklearn.metrics import accuracy_score

nb_accuracy = accuracy_score(y_test, nb_predictions)

print("Naive Bayes Accuracy:", nb_accuracy)

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

nb_precision = precision_score(y_test, nb_predictions)
nb_recall = recall_score(y_test, nb_predictions)
nb_f1 = f1_score(y_test, nb_predictions)

print("Precision:", nb_precision)
print("Recall:", nb_recall)
print("F1-score:", nb_f1)

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(y_test, nb_predictions)) 

In [ ]:
from sklearn.metrics import confusion_matrix

nb_cm = confusion_matrix(y_test, nb_predictions)

print(nb_cm)

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

ConfusionMatrixDisplay(
    confusion_matrix=nb_cm,
    display_labels=["Ham", "Spam"]
).plot()

plt.title("Naive Bayes Confusion Matrix")
plt.show()

## 11. Logistic Regression Model

Logistic Regression is trained using the same TF-IDF unigram features.

This provides a second model for comparison with Naive Bayes.

In [ ]:
from sklearn.linear_model import LogisticRegression

In [ ]:
lr_model = LogisticRegression(max_iter=1000)

In [ ]:
lr_model.fit(X_train_tfidf, y_train)

In [ ]:
lr_predictions = lr_model.predict(X_test_tfidf)

In [ ]:
lr_accuracy = accuracy_score(y_test, lr_predictions)
lr_precision = precision_score(y_test, lr_predictions)
lr_recall = recall_score(y_test, lr_predictions)
lr_f1 = f1_score(y_test, lr_predictions)

print("Logistic Regression Accuracy:", lr_accuracy)
print("Precision:", lr_precision)
print("Recall:", lr_recall)
print("F1-score:", lr_f1)

In [ ]:
print(classification_report(y_test, lr_predictions))

In [ ]:
lr_cm = confusion_matrix(y_test, lr_predictions)

print(lr_cm)

In [ ]:
ConfusionMatrixDisplay(
    confusion_matrix=lr_cm,
    display_labels=["Ham", "Spam"]
).plot()

plt.title("Logistic Regression Confusion Matrix")
plt.show()

## 12. Model Comparison

In [ ]:
results = pd.DataFrame({
    "Model": ["Naive Bayes", "Logistic Regression"],
    "Accuracy": [nb_accuracy, lr_accuracy],
    "Precision": [nb_precision, lr_precision],
    "Recall": [nb_recall, lr_recall],
    "F1-score": [nb_f1, lr_f1]
})

print(results)

## 13. Error Analysis

In [ ]:
import pandas as pd

error_analysis = pd.DataFrame({
    "message": X_test,
    "actual": y_test,
    "predicted": lr_predictions
})

errors = error_analysis[
    error_analysis["actual"] != error_analysis["predicted"]
]

print("Total incorrect predictions:", len(errors))
print(errors)

In [ ]:
false_negatives = error_analysis[
    (error_analysis["actual"] == 1) &
    (error_analysis["predicted"] == 0)
]

false_positives = error_analysis[
    (error_analysis["actual"] == 0) &
    (error_analysis["predicted"] == 1)
]

print("False Negatives:", len(false_negatives))
print("False Positives:", len(false_positives))

In [ ]:
print(false_negatives[["message"]].to_string(index=False))

In [ ]:
print(false_positives[["message"]].to_string(index=False))

### Error Analysis Findings

The baseline model produced:

- **Total incorrect predictions:** 39
- **False Negatives:** 38
- **False Positives:** 1

Common patterns observed among false negatives included:

- Abbreviations
- Unusual spelling
- Phone numbers and numeric codes
- Promotional wording
- Conversational-looking spam
- Short messages
- Variations of spam vocabulary

These patterns indicate that some spam messages can resemble normal conversations or use wording that is less strongly associated with spam.

### False Positive Example

One false positive was:

> "Waiting for your call"

The actual label was **HAM**, but the baseline model predicted it as **SPAM**.

## 14. TF-IDF Bigram Experiment

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

bigram_vectorizer = TfidfVectorizer(ngram_range=(1, 2))

X_train_bigram = bigram_vectorizer.fit_transform(X_train)
X_test_bigram = bigram_vectorizer.transform(X_test)

print("Training shape:", X_train_bigram.shape)
print("Testing shape:", X_test_bigram.shape)

In [ ]:
from sklearn.linear_model import LogisticRegression

lr_bigram_model = LogisticRegression(max_iter=1000)

lr_bigram_model.fit(X_train_bigram, y_train)

lr_bigram_predictions = lr_bigram_model.predict(X_test_bigram)

print("Experimental model trained successfully!")

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

bigram_accuracy = accuracy_score(y_test, lr_bigram_predictions)
bigram_precision = precision_score(y_test, lr_bigram_predictions)
bigram_recall = recall_score(y_test, lr_bigram_predictions)
bigram_f1 = f1_score(y_test, lr_bigram_predictions)

print("EXPERIMENTAL MODEL: Logistic Regression + TF-IDF (1,2)")
print("-----------------------------------------------------")
print(f"Accuracy : {bigram_accuracy * 100:.2f}%")
print(f"Precision: {bigram_precision * 100:.2f}%")
print(f"Recall   : {bigram_recall * 100:.2f}%")
print(f"F1-score : {bigram_f1 * 100:.2f}%")

In [ ]:
bigram_errors = (y_test != lr_bigram_predictions).sum()

print("Baseline errors     :", (y_test != lr_predictions).sum())
print("Experimental errors :", bigram_errors)

In [ ]:
comparison = pd.DataFrame({
    "Model": [
        "Baseline TF-IDF (1,1)",
        "Experimental TF-IDF (1,2)"
    ],
    "Accuracy": [
        lr_accuracy,
        bigram_accuracy
    ],
    "Precision": [
        lr_precision,
        bigram_precision
    ],
    "Recall": [
        lr_recall,
        bigram_recall
    ],
    "F1-score": [
        lr_f1,
        bigram_f1
    ]
})

print(comparison)

### Experiment 1 — TF-IDF Unigrams + Bigrams

#### Result

- **Accuracy:** 95.55%
- **Precision:** 98.85%
- **Recall:** 66.56%
- **F1-score:** 78.90%

The bigram configuration performed worse than the unigram baseline across the main performance metrics. Therefore, the unigram representation was retained.

## 15. Decision Threshold Analysis

Different decision thresholds were evaluated using the baseline Logistic Regression model.

The purpose of this experiment was to understand the trade-off between precision and recall.

In [ ]:
spam_probabilities = lr_model.predict_proba(X_test_tfidf)[:, 1]

thresholds = [0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60]

threshold_results = []

for threshold in thresholds:

    threshold_predictions = (
        spam_probabilities >= threshold
    ).astype(int)

    accuracy = accuracy_score(
        y_test,
        threshold_predictions
    )

    precision = precision_score(
        y_test,
        threshold_predictions,
        zero_division=0
    )

    recall = recall_score(
        y_test,
        threshold_predictions,
        zero_division=0
    )

    f1 = f1_score(
        y_test,
        threshold_predictions,
        zero_division=0
    )

    tn, fp, fn, tp = confusion_matrix(
        y_test,
        threshold_predictions
    ).ravel()

    threshold_results.append([
        threshold,
        accuracy,
        precision,
        recall,
        f1,
        fp,
        fn
    ])

In [ ]:
threshold_df = pd.DataFrame(
    threshold_results,
    columns=[
        "Threshold",
        "Accuracy",
        "Precision",
        "Recall",
        "F1-score",
        "False Positives",
        "False Negatives"
    ]
)

print(threshold_df)

### Threshold Analysis Result

A threshold of **0.30** produced:

- **Accuracy:** 97.49%
- **Precision:** 94.87%
- **Recall:** 84.73%
- **F1-score:** 89.52%
- **False Positives:** 6
- **False Negatives:** 20

Although the lower threshold improved recall and F1-score, it increased false positives and reduced precision.

## 16. Hyperparameter Tuning

In [ ]:
from sklearn.linear_model import LogisticRegression

lr_tuned_model = LogisticRegression(
    C=2.0,
    max_iter=1000
)

lr_tuned_model.fit(X_train_tfidf, y_train)

lr_tuned_predictions = lr_tuned_model.predict(X_test_tfidf)

print("Tuned Logistic Regression trained successfully.")

In [ ]:
tuned_accuracy = accuracy_score(y_test, lr_tuned_predictions)
tuned_precision = precision_score(y_test, lr_tuned_predictions)
tuned_recall = recall_score(y_test, lr_tuned_predictions)
tuned_f1 = f1_score(y_test, lr_tuned_predictions)

print("TUNED LOGISTIC REGRESSION")
print("------------------------")
print(f"Accuracy : {tuned_accuracy * 100:.2f}%")
print(f"Precision: {tuned_precision * 100:.2f}%")
print(f"Recall   : {tuned_recall * 100:.2f}%")
print(f"F1-score : {tuned_f1 * 100:.2f}%")

In [ ]:
tuned_cm = confusion_matrix(y_test, lr_tuned_predictions)

print("TUNED MODEL CONFUSION MATRIX")
print("----------------------------")
print(tuned_cm)

tn, fp, fn, tp = tuned_cm.ravel()

print("\nTrue Negatives :", tn)
print("False Positives:", fp)
print("False Negatives:", fn)
print("True Positives :", tp)

### Experiment 2 — Logistic Regression Hyperparameter Tuning

The tuned Logistic Regression model used **C = 2.0**.

#### Result

- **Accuracy:** 97.29%
- **Precision:** 99.05%
- **Recall:** 79.39%
- **F1-score:** 88.14%
- **False Positives:** 1
- **False Negatives:** 27

This configuration improved the baseline model while maintaining only one false positive.

### Experiment Conclusion

The experiments showed that different configurations produce different precision-recall trade-offs.

The final model was selected based on the overall balance between precision, recall, F1-score and false positives rather than accuracy alone.

## 17. Final Model

The final selected model is a Logistic Regression classifier trained on TF-IDF unigram features.

- **Model:** Logistic Regression
- **TF-IDF:** Unigrams
- **C:** 2.0
- **Decision Threshold:** 0.50

In [ ]:
final_model = lr_tuned_model

print("Final model set successfully.")
print("Model: Logistic Regression")
print("C:", final_model.C)
print("Threshold: 0.50")

## 18. New Message Prediction

The final trained model can be used to classify new, unseen SMS messages as either **HAM** or **SPAM**.

The prediction process is:

1. Validate the input message.
2. Clean the message.
3. Transform it using the trained TF-IDF vectorizer.
4. Pass the transformed message to the final Logistic Regression model.
5. Predict HAM or SPAM.
6. Display the prediction probability.

The trained TF-IDF vectorizer and final model are reused without fitting them again on the new message.

In [ ]:
def predict_message(message):
    if not message or not message.strip():
        print("Please enter a message.")
        return

    cleaned = clean_text(message)

    if not cleaned.strip():
        print("Please enter a meaningful message.")
        return

    tfidf_message = vectorizer.transform([cleaned])

    prediction = final_model.predict(tfidf_message)[0]
    probabilities = final_model.predict_proba(tfidf_message)[0]

    if prediction == 0:
        result = "HAM"
        probability = probabilities[0]
        label = "Ham"
    else:
        result = "SPAM"
        probability = probabilities[1]
        label = "Spam"

    print("Prediction:", result)
    print(f"{label} Probability: {probability * 100:.2f}%")

### New Message Testing

Several manually selected SMS messages are used to verify the behaviour of the final model on unseen messages.

In [ ]:
test_messages = [
    "Hey, are you coming to college tomorrow?",
    "Congratulations! You have won a free prize! Call now!",
    "Can you send me the notes when you get home?",
    "URGENT! Claim your special reward today before it expires!",
    "What time should we meet at the bus stop?"
]

for message in test_messages:
    print("Message:", message)
    predict_message(message)
    print("-" * 60)

## Dataset Information

### Original Dataset

The original SMS Spam Collection dataset contained:

- **Total messages:** 5,572
- **HAM messages:** 4,825
- **SPAM messages:** 747
- **Duplicate records:** 403

## Final Prediction Verification

The final Logistic Regression model with C=2.0 was tested using several new SMS messages to verify its prediction behaviour.

The final model correctly classified all five manually selected test messages.

The urgent reward message received a relatively lower SPAM probability of 58.52%, but it was still correctly classified as SPAM.

Model probability should be interpreted as the classifier's estimated confidence for its prediction and not as a guarantee that the prediction is correct.

## Final Results

The final selected model is:

- **Model:** Logistic Regression
- **TF-IDF Representation:** Unigrams
- **Hyperparameter:** C = 2.0
- **Decision Threshold:** 0.50

### Final Performance

| Metric | Result |
|---|---:|
| Accuracy | 97.29% |
| Precision | 99.05% |
| Recall | 79.39% |
| F1-score | 88.14% |
| False Positives | 1 |
| False Negatives | 27 |

### Confusion Matrix

The final model produced the following confusion matrix:

```text
[[902, 1],
 [27, 104]]
```